# Pipeline ETL Local con PySpark

Este notebook implementa el pipeline ETL local según los requerimientos solicitados:
1. Inicialización de `SparkSession.builder.master("local[*]")`.
2. Lectura y perfilado exhaustivo de datos (`@data/`): claves, nulos, tipos, referencias y detección de anomalías.
3. Limpieza y unión de tablas calculando el importe neto: `quantity * unit_price * (1 - discount_pct)`.
4. Ventas netas por mes, país, canal y categoría con Spark SQL y DataFrame API (sin duplicar importes en los joins).
5. Reto incremental e idempotencia: selección de la versión más reciente por `order_id` según `updated_at`.
6. Persistencia en formato Parquet bajo `output/` y ejecución de controles de calidad (Data Quality Checks).

In [1]:
import os
import sys

# Configuración local en Windows para compatibilidad Hadoop/Winutils
project_dir = os.path.abspath('..')
hadoop_dir = os.path.join(project_dir, 'hadoop')
if os.path.exists(hadoop_dir):
    os.environ['HADOOP_HOME'] = hadoop_dir
    os.environ['PATH'] = os.path.join(hadoop_dir, 'bin') + ';' + os.environ.get('PATH', '')

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, countDistinct, when, isnan, min, max, avg, sum as spark_sum,
    round, regexp_replace, try_to_timestamp, coalesce, date_format, lit,
    row_number, desc, expr
)
from pyspark.sql.window import Window

# Inicializar SparkSession local
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('ETL_Notebook_PySpark') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic') \
    .getOrCreate()

print('Spark Version:', spark.version)
print('Spark Master:', spark.conf.get('spark.master'))

Spark Version: 4.3.0
Spark Master: local[*]


## 1. Carga de Data desde `data/`

In [2]:
data_dir = '../data'
clientes = spark.read.option('header', 'true').csv(os.path.join(data_dir, 'clientes.csv'))
pedidos = spark.read.option('header', 'true').csv(os.path.join(data_dir, 'pedidos.csv'))
detalle = spark.read.option('header', 'true').csv(os.path.join(data_dir, 'detalle_pedidos.csv'))
productos = spark.read.option('header', 'true').csv(os.path.join(data_dir, 'productos.csv'))
pedidos_inc = spark.read.option('header', 'true').csv(os.path.join(data_dir, 'pedidos_incrementales.csv'))
detalle_inc = spark.read.option('header', 'true').csv(os.path.join(data_dir, 'detalle_incremental.csv'))

print('Data cargada exitosamente.')
for name, df in [('clientes', clientes), ('pedidos', pedidos), ('detalle', detalle), ('productos', productos), ('pedidos_inc', pedidos_inc), ('detalle_inc', detalle_inc)]:
    print(f'  {name:15} | Filas: {df.count():6} | Columnas: {len(df.columns)}')

Data cargada exitosamente.
  clientes        | Filas:   1001 | Columnas: 7
  pedidos         | Filas:  20001 | Columnas: 6
  detalle         | Filas:  40026 | Columnas: 5
  productos       | Filas:    100 | Columnas: 5
  pedidos_inc     | Filas:    101 | Columnas: 6
  detalle_inc     | Filas:    200 | Columnas: 5


## 2. Perfilado de Datos y Detección de Anomalías
Evaluamos unicidad de claves, valores nulos, integridad referencial y anomalías en datos numéricos/fechas.

In [3]:
# 2.1 Unicidad de Claves Primarias y Duplicados
print('--- Unicidad de Claves Primarias ---')
ped_dups = pedidos.groupBy('order_id').count().filter('count > 1')
print('Pedidos con order_id duplicado:')
ped_dups.show()

cli_dups = clientes.groupBy('customer_id').count().filter('count > 1')
print('Clientes con customer_id duplicado:')
cli_dups.show()

# 2.2 Integridad Referencial (Registros Huérfanos)
print('--- Integridad Referencial ---')
orphan_orders = detalle.join(pedidos, on='order_id', how='left_anti')
print(f'Detalle con order_id inexistente: {orphan_orders.count()}')
orphan_orders.show()

orphan_prods = detalle.join(productos, on='product_id', how='left_anti')
print(f'Detalle con product_id inexistente: {orphan_prods.count()}')
orphan_prods.show()

orphan_custs = pedidos.join(clientes, on='customer_id', how='left_anti')
print(f'Pedidos con customer_id inexistente o nulo: {orphan_custs.count()}')
orphan_custs.show()

# 2.3 Formatos y Outliers Numéricos / Fechas
print('--- Outliers Numéricos y Formatos ---')
det_parsed = detalle \
    .withColumn('qty_int', col('quantity').cast('int')) \
    .withColumn('disc_flt', regexp_replace(col('discount_pct'), ',', '.').cast('double'))

print('Detalle con quantity <= 0:')
det_parsed.filter('qty_int <= 0').show()

print('Detalle con discount_pct > 1.0 (120%):')
det_parsed.filter('disc_flt > 1.0').show()

prod_parsed = productos.withColumn('price_flt', regexp_replace(col('unit_price'), ',', '.').cast('double'))
print('Productos con precio <= 0 o nulo:')
prod_parsed.filter('price_flt <= 0 or price_flt is null').show()

ped_dates = pedidos.withColumn('ts_eval', try_to_timestamp(col('order_ts')))
print('Pedidos con fecha corrupta en order_ts (mes 99):')
ped_dates.filter(col('ts_eval').isNull()).show()

--- Unicidad de Claves Primarias ---
Pedidos con order_id duplicado:
+--------+-----+
|order_id|count|
+--------+-----+
|  O00012|    2|
+--------+-----+

Clientes con customer_id duplicado:
+-----------+-----+
|customer_id|count|
+-----------+-----+
|      C0020|    2|
+-----------+-----+

--- Integridad Referencial ---
Detalle con order_id inexistente: 1
+--------+--------+----------+--------+------------+
|order_id| line_id|product_id|quantity|discount_pct|
+--------+--------+----------+--------+------------+
|  O99999|L0000003|      P028|       3|           0|
+--------+--------+----------+--------+------------+

Detalle con product_id inexistente: 1
+----------+--------+--------+--------+------------+
|product_id| line_id|order_id|quantity|discount_pct|
+----------+--------+--------+--------+------------+
|      P999|L0000004|  O00003|       2|         0,1|
+----------+--------+--------+--------+------------+

Pedidos con customer_id inexistente o nulo: 2
+-----------+--------+---

### Resumen de Anomalías y Estrategia de Tratamiento
1. **Separador decimal con coma y tipos**: `discount_pct` y `unit_price` usan coma (`0,05`, `29,85`). Tratamiento: `regexp_replace(',', '.')` y casteo a `DoubleType` / `IntegerType`.
2. **Valores numéricos de negocio inválidos (Outliers)**: `quantity <= 0` (0 y -2), `discount_pct = 1,2` (> 100%), `unit_price` negativo (`-12.5`) o nulo. Tratamiento: filtrar líneas no válidas para mantener integridad contable.
3. **Duplicados en Claves Primarias**: `customer_id = 'C0020'` en clientes y `order_id = 'O00012'` en pedidos. Tratamiento: `dropDuplicates([PK])` para eliminar duplicación y prevenir join fan-out.
4. **Violación de integridad referencial**: Huérfanos en `detalle_pedidos` y `pedidos`. Tratamiento: `INNER JOIN` con dimensiones limpias.
5. **Timestamp corrupto y typos en estado**: `order_ts = '2026-99-31'` en pedido `O00025` y `status = 'deliverd'`. Tratamiento: rescate con `coalesce(try_to_timestamp(order_ts), try_to_timestamp(updated_at))` y normalización del estado a `'delivered'`.

## 3 & 5. Limpieza, Resolución Incremental (Idempotencia / CDC) y Enriquecimiento
- Unión de base e incremental con deduplicación por `order_id` priorizando la versión más reciente según `updated_at`.
- Joins N:1 garantizados sin duplicar importes.
- Cálculo del importe neto: `quantity * unit_price * (1 - discount_pct)`.

In [4]:
# 1. Clientes limpios
clientes_clean = clientes.dropDuplicates(['customer_id']).filter(col('customer_id').isNotNull())

# 2. Productos limpios
productos_clean = productos \
    .withColumn('unit_price', regexp_replace(col('unit_price'), ',', '.').cast('double')) \
    .filter(col('unit_price').isNotNull() & (col('unit_price') > 0.0)) \
    .dropDuplicates(['product_id'])

# 3. Pedidos: Ingesta incremental con Window row_number() para CDC idempotente
pedidos_union = pedidos.unionByName(pedidos_inc) \
    .withColumn('ts_updated', coalesce(try_to_timestamp(col('updated_at')), try_to_timestamp(col('order_ts')))) \
    .withColumn('ts_order', coalesce(try_to_timestamp(col('order_ts')), try_to_timestamp(col('updated_at')))) \
    .withColumn('status_clean', when(col('status') == 'deliverd', lit('delivered')).otherwise(col('status'))) \
    .filter(col('customer_id').isNotNull())

w_orders = Window.partitionBy('order_id').orderBy(col('ts_updated').desc())
pedidos_clean = pedidos_union \
    .withColumn('rn', row_number().over(w_orders)) \
    .filter(col('rn') == 1) \
    .drop('rn') \
    .withColumn('mes', date_format(col('ts_order'), 'yyyy-MM')) \
    .filter(col('mes').isNotNull())

# 4. Detalle de pedidos: Union incremental y reglas de negocio
detalle_union = detalle.unionByName(detalle_inc) \
    .withColumn('quantity', col('quantity').cast('int')) \
    .withColumn('discount_pct', regexp_replace(col('discount_pct'), ',', '.').cast('double')) \
    .filter(col('quantity') > 0) \
    .filter((col('discount_pct') >= 0.0) & (col('discount_pct') <= 1.0)) \
    .dropDuplicates(['line_id'])

# 5. Join limpio e Importe Neto
lineas_enriquecidas = detalle_union \
    .join(pedidos_clean.select('order_id', 'customer_id', 'mes', 'channel', 'status_clean'), on='order_id', how='inner') \
    .join(clientes_clean.select('customer_id', 'country'), on='customer_id', how='inner') \
    .join(productos_clean.select('product_id', 'category', 'unit_price'), on='product_id', how='inner') \
    .withColumn(
        'net_amount',
        round(col('quantity') * col('unit_price') * (lit(1.0) - col('discount_pct')), 4)
    )

print(f'Total de líneas limpias procesadas: {lineas_enriquecidas.count()}')
lineas_enriquecidas.select('line_id', 'order_id', 'category', 'country', 'channel', 'quantity', 'unit_price', 'discount_pct', 'net_amount').show(5)

Total de líneas limpias procesadas: 39389
+---------+--------+-----------+-------+-------+--------+----------+------------+----------+
|  line_id|order_id|   category|country|channel|quantity|unit_price|discount_pct|net_amount|
+---------+--------+-----------+-------+-------+--------+----------+------------+----------+
|IL0000001|  O20001|electronics|     CO|    web|       2|     29.85|         0.0|      59.7|
|IL0000002|  O20001|      books|     CO|    web|       3|     151.3|        0.05|   431.205|
|IL0000003|  O20002|       home|     MX| mobile|       3|      47.2|        0.05|    134.52|
|IL0000004|  O20002|electronics|     MX| mobile|       4|    168.65|         0.1|    607.14|
|IL0000005|  O20003|     sports|     CL|  store|       4|     64.55|         0.1|    232.38|
+---------+--------+-----------+-------+-------+--------+----------+------------+----------+
only showing top 5 rows


## 4. Ventas Netas por Mes, País, Canal y Categoría
Cálculo con la API de DataFrames y verificación con Spark SQL.

In [5]:
# DataFrame API
ventas_agg = lineas_enriquecidas.groupBy('mes', 'country', 'channel', 'category') \
    .agg(
        round(spark_sum('net_amount'), 2).alias('ventas_netas'),
        count('line_id').alias('num_lineas'),
        spark_sum('quantity').alias('unidades_vendidas')
    ) \
    .orderBy('mes', 'country', 'channel', 'category')

# Spark SQL (Equivalencia)
lineas_enriquecidas.createOrReplaceTempView('view_lineas')
ventas_sql = spark.sql('''
    SELECT mes, country, channel, category, ROUND(SUM(net_amount), 2) AS ventas_netas, COUNT(line_id) AS num_lineas
    FROM view_lineas
    GROUP BY mes, country, channel, category
    ORDER BY mes, country, channel, category
''')

ventas_agg.show(15, truncate=False)

+-------+-------+-------+-----------+------------+----------+-----------------+
|mes    |country|channel|category   |ventas_netas|num_lineas|unidades_vendidas|
+-------+-------+-------+-----------+------------+----------+-----------------+
|2026-01|CL     |mobile |books      |1101398.96  |434       |1292             |
|2026-01|CL     |mobile |electronics|948243.87   |417       |1261             |
|2026-01|CL     |mobile |home       |1016902.25  |437       |1297             |
|2026-01|CL     |mobile |sports     |1120925.81  |455       |1371             |
|2026-01|CL     |store  |books      |1015411.42  |407       |1232             |
|2026-01|CL     |store  |electronics|882199.87   |392       |1166             |
|2026-01|CL     |store  |home       |1052104.16  |443       |1313             |
|2026-01|CL     |store  |sports     |1098003.25  |459       |1333             |
|2026-01|CL     |web    |books      |1113534.5   |443       |1360             |
|2026-01|CL     |web    |electronics|101

## 6. Persistencia en Parquet (`output/`) y Controles de Calidad
Guardamos con modo `overwrite` (idempotencia) y validamos la consistencia con 2 controles de calidad automáticos.

In [6]:
output_path = '../output/ventas_netas_agregadas.parquet'
ventas_agg.write.mode('overwrite').parquet(output_path)
print('Salida persistida en Parquet exitosamente.')

# Leer archivo guardado
persisted_df = spark.read.parquet(output_path)

# Control 1: Reconciliación de totales (Zero Duplication)
total_line_amount = lineas_enriquecidas.select(spark_sum('net_amount')).collect()[0][0]
total_agg_amount = persisted_df.select(spark_sum('ventas_netas')).collect()[0][0]
diff = abs(total_line_amount - total_agg_amount)

print(f'[CONTROL 1] Reconciliación de Masa Financiera:')
print(f'  - Total Líneas:   ${total_line_amount:,.2f}')
print(f'  - Total Agregado: ${total_agg_amount:,.2f}')
print(f'  - Discrepancia:   ${diff:.4f}')
assert diff < 1.0, f'Error: Duplicación en joins detectada: diff={diff}'
print('>>> RESULTADO CONTROL 1: APROBADO (0 Duplicación en Joins)')

# Control 2: Unicidad y Completitud
total_rows = persisted_df.count()
distinct_keys = persisted_df.select('mes', 'country', 'channel', 'category').distinct().count()
null_count = persisted_df.filter(
    col('mes').isNull() | col('country').isNull() | col('channel').isNull() | col('category').isNull()
).count()
non_positive = persisted_df.filter(col('ventas_netas') <= 0).count()

print(f'[CONTROL 2] Unicidad, Completitud y Validez:')
print(f'  - Total Registros:        {total_rows}')
print(f'  - Claves Únicas:          {distinct_keys}')
print(f'  - Nulos en Claves:        {null_count}')
print(f'  - Ventas Netas <= 0:      {non_positive}')
assert total_rows == distinct_keys, 'Error: Claves duplicadas.'
assert null_count == 0, 'Error: Valores nulos encontrados.'
assert non_positive == 0, 'Error: Ventas netas no positivas.'
print('>>> RESULTADO CONTROL 2: APROBADO (Granularidad y Dominio Coherente)')

Salida persistida en Parquet exitosamente.
[CONTROL 1] Reconciliación de Masa Financiera:
  - Total Líneas:   $95,715,339.93
  - Total Agregado: $95,715,340.00
  - Discrepancia:   $0.0650
>>> RESULTADO CONTROL 1: APROBADO (0 Duplicación en Joins)
[CONTROL 2] Unicidad, Completitud y Validez:
  - Total Registros:        208
  - Claves Únicas:          208
  - Nulos en Claves:        0
  - Ventas Netas <= 0:      0
>>> RESULTADO CONTROL 2: APROBADO (Granularidad y Dominio Coherente)
